# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravindidhananjana/Internship-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import pandas as pd
import numpy as np

# 1. Setup HF Token and DuckDB connection
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# 2. Build feature vector strictly from historical window (March 1-15, 2026)
features_df = con.sql(f"""
    WITH windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Outcome window (March 16-31, 2026)
            SUM(CASE WHEN f.report_date > DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_last15,

            -- Historical Features (Knowable BEFORE decision moment: March 1-15, 2026)
            SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
            AVG(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_avg_prev,
            STDDEV_SAMP(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_std_prev

        FROM {TABLES['fact_daily']} f
        WHERE f.report_date >= '2026-03-01' AND f.report_date <= '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_prev15 >= 10
    )
    SELECT * FROM windowed
""").df()

# Missing value handling for single-record standard deviation
features_df['has_pos_std'] = features_df['pos_std_prev'].notna().astype(int)
features_df['pos_std_prev'] = features_df['pos_std_prev'].fillna(0)

# Merge query-level context signals
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           MAX(impressions_90d) / NULLIF(SUM(impressions_90d), 0) AS top_query_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

features_df = features_df.merge(qsignals, on='content_hash_id', how='left').fillna(0)

# Target label definition: impression decline >= 20%
features_df['is_declining'] = (features_df['imp_last15'] < 0.8 * features_df['imp_prev15']).astype(int)

print(f"Feature matrix built successfully: {features_df.shape[0]} rows, {features_df.shape[1]} columns")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature matrix built successfully: 120513 rows, 10 columns


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


imp_prev15: Sum of search impressions over the historical observation window (March 1–15). No missing values. Available before the decision moment.

pos_avg_prev: Average Google Search position rank during March 1–15. Missing values occur if no rank data exists. Available before the decision moment.

pos_std_prev: Standard deviation of daily position ranks. Missing for items with 1 record; handled by filling 0 and adding indicator flag has_pos_std. Available before the decision moment.

visible_queries: Count of distinct search queries for which content appeared. Left-joined from historical query log; non-matches filled with 0. Available before the decision moment.

top_query_share: Impression concentration ratio of the #1 query relative to total query impressions. Filled with 0 for unindexed items. Available before the decision moment.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Honest Feature Set
honest_features = ['imp_prev15', 'pos_avg_prev', 'pos_std_prev', 'has_pos_std', 'visible_queries', 'top_query_share']
X_honest = features_df[honest_features]
y = features_df['is_declining']

model_honest = RandomForestClassifier(random_state=42, n_estimators=50).fit(X_honest, y)
acc_honest = accuracy_score(y, model_honest.predict(X_honest))

# 2. Inject Future Outcome Signal (Deliberate Target Leakage)
X_leaked = X_honest.copy()
X_leaked['leaked_future_signal'] = features_df['imp_last15']

model_leaked = RandomForestClassifier(random_state=42, n_estimators=50).fit(X_leaked, y)
acc_leaked = accuracy_score(y, model_leaked.predict(X_leaked))

print(f"Honest Feature Set Accuracy:   {acc_honest:.4f}")
print(f"Leaked Feature Set Accuracy:   {acc_leaked:.4f}")
print(f"Leakage Impact (Jump in Acc):  +{(acc_leaked - acc_honest):.4f}")


Honest Feature Set Accuracy:   0.9995
Leaked Feature Set Accuracy:   1.0000
Leakage Impact (Jump in Acc):  +0.0005


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Privacy Verification:

All client IDs (client_hash_id) and content IDs (content_hash_id) are pseudonymous 128-bit hashes.

No raw search query text, searcher IP addresses, user identifiers, or domain names are contained within the feature set.

Features represent aggregate numeric counts and ranks over 15-day intervals, eliminating any risk of individual re-identification.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.